# Mão na Roda — Fine-tuning do DeBERTinha
Diagnóstico Preliminar de Falhas Automotivas Baseado em IA

Notebook equivalente aos do BERTimbau e do BERTabaporu, trocando o modelo
base para o **DeBERTinha** (`sagui-nlp/debertinha-ptbr-xsmall`).

**Sobre o modelo:** diferente do BERTimbau e do BERTabaporu (ambos
arquitetura BERT), o DeBERTinha é baseado em **DeBERTa-v2**, adaptado para
o português a partir do DeBERTa-v3 XSmall (Campos, Lotufo e Nogueira,
arXiv:2309.16844). Tem apenas ~40M de parâmetros — bem menor que os
~110M do BERTimbau/BERTabaporu — mas o artigo original reporta desempenho
superior ao BERTimbau-Large em tarefas de NER e análise de sentimento,
apesar do tamanho reduzido. Isso torna esse experimento um teste de
hipótese diferente dos anteriores: não é mais "qual domínio de pré-treino
generaliza melhor", e sim "uma arquitetura mais eficiente e compacta
consegue igualar um modelo maior nesta tarefa de classificação de
falhas automotivas?".

**Diferença técnica a observar:** o DeBERTinha usa tokenizer SentencePiece
(arquitetura DeBERTa-v2), diferente do WordPiece usado por BERTimbau e
BERTabaporu. Isso é tratado automaticamente pelo `AutoTokenizer`, mas pode
gerar uma contagem de tokens por relato ligeiramente diferente para o
mesmo texto — vale registrar isso no TCC se a distribuição de tokens
truncados em `MAX_LENGTH=64` divergir muito entre os modelos.

**Antes de rodar:** Ambiente de execução → Alterar tipo de ambiente de
execução → GPU (T4).

## Passo 1 — Setup e verificação do ambiente

Inclui, logo no início, a remoção preventiva do `torchvision` — bug de
ambiente já identificado no notebook do BERTabaporu (a lib `datasets`
quebra ao tentar importar `VideoReader` de versões recentes do
`torchvision`, algo que este projeto não usa em nenhum momento).

In [1]:
# ============================================================
# PASSO 1 — Setup e verificação do ambiente
# Projeto Mão na Roda — Fine-tuning do DeBERTinha
# ============================================================

# --- 1.0 Corrige bug de ambiente conhecido (datasets x torchvision) ---
# O projeto não usa nenhum recurso de visão computacional; remover o
# torchvision evita o ImportError "cannot import name 'VideoReader'"
# que ocorre ao formatar datasets como tensores PyTorch.
!pip uninstall -y torchvision -q

# --- 1.1 Instala as dependências ---
!pip install -q transformers datasets accelerate scikit-learn pandas openpyxl sentencepiece

# --- 1.2 Verifica se a GPU está disponível ---
import torch
print("Torch:", torch.__version__)
print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  ATENÇÃO: nenhuma GPU detectada. Vá em 'Ambiente de execução' > "
          "'Alterar tipo de ambiente de execução' e selecione GPU antes de continuar.")

# --- 1.3 Constantes do projeto ---
# Único ponto que muda em relação aos notebooks anteriores: o checkpoint.
MODEL_CHECKPOINT = "sagui-nlp/debertinha-ptbr-xsmall"  # DeBERTinha (Campos, Lotufo e Nogueira, arXiv:2309.16844)
SEED = 42
NUM_LABELS = 10  # as 10 classes de falha do projeto
MAX_LENGTH = 64  # mesmo valor usado nos notebooks anteriores, para comparação justa

LABEL_NAMES = [
    "Bateria/Sistema Elétrico",       # 0
    "Sistema de Freios",              # 1
    "Superaquecimento do Motor",      # 2
    "Suspensão/Amortecedores",        # 3
    "Sistema de Transmissão/Câmbio",  # 4
    "Vazamento de Óleo",              # 5
    "Sistema de Arrefecimento",       # 6
    "Pneu/Roda",                      # 7
    "Sistema de Injeção/Combustível", # 8
    "Sistema de Escapamento",         # 9
]

import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print()
print("Setup concluído. Modelo base:", MODEL_CHECKPOINT)

# ============================================================
# --- 1.4 Carregando os arquivos de split (via Google Drive) ---
# ============================================================
# Reutiliza os MESMOS splits dos notebooks anteriores — essencial para
# que a comparação entre os 3 modelos seja válida: todos precisam ver
# exatamente os mesmos exemplos de treino/validação/teste.

from google.colab import drive
drive.mount('/content/drive')

CAMINHO_BASE = '/content/drive/MyDrive/Agente Mecânico/notebooks/Finetuning - 3000 relatos/3 Splits - Finetuning/Splits'

import pandas as pd
import os

CONFIGS = ["80_10_10", "70_15_15", "60_20_20"]

splits = {}
for tag in CONFIGS:
    splits[tag] = {
        "treino": pd.read_csv(os.path.join(CAMINHO_BASE, f"split_treino_{tag}.csv")),
        "val":    pd.read_csv(os.path.join(CAMINHO_BASE, f"split_val_{tag}.csv")),
        "teste":  pd.read_csv(os.path.join(CAMINHO_BASE, f"split_teste_{tag}.csv")),
    }
    n_tr, n_val, n_te = len(splits[tag]["treino"]), len(splits[tag]["val"]), len(splits[tag]["teste"])
    print(f"{tag}: treino={n_tr} | val={n_val} | teste={n_te} | total={n_tr+n_val+n_te}")

assert "peso_treino" in splits["80_10_10"]["treino"].columns, \
    "Coluna peso_treino não encontrada — confira se subiu o CSV certo (do v15, não uma versão antiga)."
print("\nColuna peso_treino presente. Pronto para o Passo 2 (tokenização).")

Torch: 2.11.0+cu128
GPU disponível: True
GPU: Tesla T4

Setup concluído. Modelo base: sagui-nlp/debertinha-ptbr-xsmall
Mounted at /content/drive
80_10_10: treino=2432 | val=305 | teste=304 | total=3041
70_15_15: treino=2171 | val=435 | teste=435 | total=3041
60_20_20: treino=1824 | val=608 | teste=609 | total=3041

Coluna peso_treino presente. Pronto para o Passo 2 (tokenização).


## Passo 2 — Tokenização

O tokenizer é carregado a partir do próprio `MODEL_CHECKPOINT` — igual
nos notebooks anteriores. Aqui isso é ainda mais importante: o DeBERTinha
usa um tokenizer SentencePiece bem diferente do WordPiece do BERTimbau e
do BERTabaporu, então carregar o tokenizer errado não geraria só um
resultado ruim — provavelmente quebraria de forma explícita, já que os
IDs de vocabulário nem teriam o mesmo espaço de tamanho.

In [2]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_batch(batch):
    return tokenizer(
        batch["relato"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def preparar_dataset_hf(df):
    # Converte um DataFrame pandas em um Dataset do Hugging Face,
    # já tokenizado e com as colunas 'labels' e 'weight' prontas para o Trainer.
    ds = Dataset.from_pandas(df[["relato", "falha_label", "peso_treino"]].reset_index(drop=True))
    ds = ds.map(tokenize_batch, batched=True)
    ds = ds.rename_column("falha_label", "labels")
    ds = ds.rename_column("peso_treino", "weight")
    ds = ds.remove_columns(["relato"])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels", "weight"])
    return ds

# Tokeniza as 3 configs de uma vez (usadas na comparação leve do Passo 4)
datasets_tokenizados = {}
for tag in CONFIGS:
    datasets_tokenizados[tag] = {
        "treino": preparar_dataset_hf(splits[tag]["treino"]),
        "val":    preparar_dataset_hf(splits[tag]["val"]),
        "teste":  preparar_dataset_hf(splits[tag]["teste"]),
    }
    print(f"{tag}: tokenização concluída "
          f"({len(datasets_tokenizados[tag]['treino'])} treino, "
          f"{len(datasets_tokenizados[tag]['val'])} val, "
          f"{len(datasets_tokenizados[tag]['teste'])} teste)")

# Checagem rápida: mostra um exemplo tokenizado para conferência visual
exemplo = datasets_tokenizados["80_10_10"]["treino"][0]
print("\nExemplo tokenizado:")
print("input_ids (primeiros 15):", exemplo["input_ids"][:15])
print("labels:", exemplo["labels"], "->", LABEL_NAMES[int(exemplo["labels"])])
print("weight:", exemplo["weight"])

# --- Checagem específica para DeBERTinha: quantos relatos foram truncados? ---
# Como o tokenizer é diferente (SentencePiece), a taxa de truncamento em
# MAX_LENGTH=64 pode diferir do BERTimbau/BERTabaporu — vale documentar.
def contar_truncados(df, tokenizer, max_length):
    truncados = 0
    for texto in df["relato"]:
        ids_completos = tokenizer(texto, truncation=False)["input_ids"]
        if len(ids_completos) > max_length:
            truncados += 1
    return truncados, len(df)

n_truncados, n_total = contar_truncados(splits["80_10_10"]["treino"], tokenizer, MAX_LENGTH)
print(f"\nRelatos truncados em MAX_LENGTH={MAX_LENGTH} (treino, DeBERTinha): "
      f"{n_truncados}/{n_total} ({100*n_truncados/n_total:.1f}%)")

config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.62M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Map:   0%|          | 0/2432 [00:00<?, ? examples/s]

Map:   0%|          | 0/305 [00:00<?, ? examples/s]

... (6 outputs left out)

Map:   0%|          | 0/1824 [00:00<?, ? examples/s]

Map:   0%|          | 0/608 [00:00<?, ? examples/s]

Map:   0%|          | 0/609 [00:00<?, ? examples/s]

60_20_20: tokenização concluída (1824 treino, 608 val, 609 teste)

Exemplo tokenizado:
input_ids (primeiros 15): tensor([50269,    31,  1326,    38,  1361,     4,  2869,  1079,     3,   201,
          879,    23, 13716,   133, 13293])
labels: tensor(0) -> Bateria/Sistema Elétrico
weight: tensor(0.5000)

Relatos truncados em MAX_LENGTH=64 (treino, DeBERTinha): 0/2432 (0.0%)


## Passo 3 — `WeightedTrainer` e métricas

Idêntico aos notebooks anteriores — a lógica de ponderação por
`peso_treino` é independente do modelo.

In [3]:
import torch.nn as nn
from transformers import Trainer
from sklearn.metrics import f1_score, accuracy_score, classification_report

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        weights = inputs.pop("weight").float()
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(reduction="none")
        per_example_loss = loss_fct(logits, labels)
        loss = (per_example_loss * weights).sum() / weights.sum()
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1_macro": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
    }

print("WeightedTrainer e compute_metrics definidos.")

WeightedTrainer e compute_metrics definidos.


## Passo 4 — Treino leve comparativo (3 configs)

Mesmo procedimento dos notebooks anteriores: poucas épocas, só para
comparar o F1-macro de validação entre as 3 proporções de split.

In [4]:
from transformers import AutoModelForSequenceClassification, TrainingArguments

def rodar_treino_leve(tag, epocas=3):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT, num_labels=NUM_LABELS
    )

    args = TrainingArguments(
        output_dir=f"./resultados_leve_debertinha_{tag}",
        num_train_epochs=epocas,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        seed=SEED,
        remove_unused_columns=False,   # crítico — não remover
        report_to="none",
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=datasets_tokenizados[tag]["treino"],
        eval_dataset=datasets_tokenizados[tag]["val"],
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics_val = trainer.evaluate()
    return metrics_val["eval_f1_macro"], metrics_val["eval_accuracy"]

resultados_leve = {}
for tag in CONFIGS:
    print(f"\n{'='*60}\nTreino leve — config {tag}\n{'='*60}")
    f1, acc = rodar_treino_leve(tag, epocas=3)
    resultados_leve[tag] = {"f1_macro": f1, "accuracy": acc}
    print(f"{tag}: F1-macro (val) = {f1:.4f} | acurácia (val) = {acc:.4f}")

print("\n=== Comparação final (treino leve, DeBERTinha) ===")
for tag, m in sorted(resultados_leve.items(), key=lambda x: -x[1]["f1_macro"]):
    print(f"{tag}: F1-macro={m['f1_macro']:.4f}  acc={m['accuracy']:.4f}")

# Lembrete: diferenças de F1 abaixo de ~0.01 devem ser tratadas como empate
# técnico nesta escala de dataset.


Treino leve — config 80_10_10


model.safetensors: reconstructing file:   0%|          |  0.00B /  163MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: sagui-nlp/debertinha-ptbr-xsmall
Key                 | Status  | 
--------------------+---------+-
classifier.bias     | MISSING | 
pooler.dense.bias   | MISSING | 
classifier.weight   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,2.246167,1.943626,0.306987,0.357377
2,1.678782,1.336976,0.536166,0.563934
3,1.196876,1.124728,0.642935,0.655738


... (8 outputs left out)

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: sagui-nlp/debertinha-ptbr-xsmall
Key                 | Status  | 
--------------------+---------+-
classifier.bias     | MISSING | 
pooler.dense.bias   | MISSING | 
classifier.weight   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,2.305597,2.296544,0.071469,0.146382
2,2.175395,1.989973,0.268571,0.330592
3,1.865075,1.820198,0.326336,0.386513


Training Loss,Validation Loss,Epoch,F1 Macro,Accuracy
1.865075,1.820198,3,0.326336,0.386513


60_20_20: F1-macro (val) = 0.3263 | acurácia (val) = 0.3865

=== Comparação final (treino leve, DeBERTinha) ===
80_10_10: F1-macro=0.6429  acc=0.6557
60_20_20: F1-macro=0.3263  acc=0.3865
70_15_15: F1-macro=0.2679  acc=0.3195


## Passo 5 — Escolha da configuração vencedora

In [5]:
# Ajustem esta linha de acordo com o resultado do Passo 4
CONFIG_VENCEDORA = "80_10_10"  # <-- editar se outra config vencer

print(f"Config escolhida para o treino final: {CONFIG_VENCEDORA}")
print(f"Treino: {len(datasets_tokenizados[CONFIG_VENCEDORA]['treino'])} exemplos")
print(f"Validação: {len(datasets_tokenizados[CONFIG_VENCEDORA]['val'])} exemplos")
print(f"Teste: {len(datasets_tokenizados[CONFIG_VENCEDORA]['teste'])} exemplos (fica intocado até a avaliação final)")

Config escolhida para o treino final: 80_10_10
Treino: 2432 exemplos
Validação: 305 exemplos
Teste: 304 exemplos (fica intocado até a avaliação final)


## Passo 6 — Treino completo com early stopping

Mesma estratégia dos notebooks anteriores: até 15 épocas, com early
stopping (patience=3) monitorando o F1-macro de validação.

In [6]:
from transformers import EarlyStoppingCallback

model_final = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=NUM_LABELS
)

args_final = TrainingArguments(
    output_dir="./resultados_final_debertinha",
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_strategy="epoch",
    seed=SEED,
    remove_unused_columns=False,
    report_to="none",
)

trainer_final = WeightedTrainer(
    model=model_final,
    args=args_final,
    train_dataset=datasets_tokenizados[CONFIG_VENCEDORA]["treino"],
    eval_dataset=datasets_tokenizados[CONFIG_VENCEDORA]["val"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer_final.train()

print("\nMelhor F1-macro de validação atingido:",
      trainer_final.state.best_metric)
print("Checkpoint carregado automaticamente (load_best_model_at_end=True).")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: sagui-nlp/debertinha-ptbr-xsmall
Key                 | Status  | 
--------------------+---------+-
classifier.bias     | MISSING | 
pooler.dense.bias   | MISSING | 
classifier.weight   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,2.297976,2.192914,0.139547,0.216393
2,1.795879,1.278171,0.576250,0.603279
3,1.007889,0.759088,0.775561,0.780328
4,0.539932,0.645269,0.784617,0.783607
5,0.347260,0.580076,0.841809,0.839344
6,0.230050,0.719067,0.818400,0.819672
7,0.161299,0.629848,0.839212,0.839344
8,0.123242,0.595436,0.853006,0.852459
9,0.079305,0.741552,0.838594,0.839344
10,0.064456,0.704282,0.861755,0.862295


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (8 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Melhor F1-macro de validação atingido: 0.8622318234700745
Checkpoint carregado automaticamente (load_best_model_at_end=True).


## Passo 7 — Avaliação final no conjunto de teste

In [7]:
resultado_teste = trainer_final.evaluate(datasets_tokenizados[CONFIG_VENCEDORA]["teste"])
print("=== Resultado final no TESTE (DeBERTinha) ===")
print(f"F1-macro: {resultado_teste['eval_f1_macro']:.4f}")
print(f"Acurácia: {resultado_teste['eval_accuracy']:.4f}")

# Relatório por classe — essencial para comparar com BERTimbau e BERTabaporu,
# em especial no par Superaquecimento x Arrefecimento
preds_output = trainer_final.predict(datasets_tokenizados[CONFIG_VENCEDORA]["teste"])
preds = np.argmax(preds_output.predictions, axis=-1)
labels_reais = preds_output.label_ids

print("\n=== Relatório por classe ===")
print(classification_report(labels_reais, preds, target_names=LABEL_NAMES, digits=3))

Training Loss,Validation Loss,Epoch,F1 Macro,Accuracy
0.027316,0.662397,14,0.878360,0.878289


=== Resultado final no TESTE (DeBERTinha) ===
F1-macro: 0.8784
Acurácia: 0.8783



=== Relatório por classe ===
                                precision    recall  f1-score   support

      Bateria/Sistema Elétrico      1.000     0.839     0.912        31
             Sistema de Freios      0.966     0.903     0.933        31
     Superaquecimento do Motor      0.719     0.767     0.742        30
       Suspensão/Amortecedores      0.893     0.806     0.847        31
 Sistema de Transmissão/Câmbio      0.882     0.968     0.923        31
             Vazamento de Óleo      0.966     0.933     0.949        30
      Sistema de Arrefecimento      0.828     0.800     0.814        30
                     Pneu/Roda      0.829     0.967     0.892        30
Sistema de Injeção/Combustível      0.839     0.867     0.852        30
        Sistema de Escapamento      0.903     0.933     0.918        30

                      accuracy                          0.878       304
                     macro avg      0.882     0.878     0.878       304
                  weighted avg  

## Passo 8 — Salvar o modelo

Mesma recomendação de sempre: salvar direto no Google Drive, não só no
disco efêmero do Colab.

In [8]:
from google.colab import drive
drive.mount('/content/drive')

CAMINHO_SALVAR = '/content/drive/MyDrive/Agente Mecânico/modelos/debertinha_mao_na_roda/'

trainer_final.save_model(CAMINHO_SALVAR)
tokenizer.save_pretrained(CAMINHO_SALVAR)

print(f"Modelo e tokenizer salvos em: {CAMINHO_SALVAR}")
print("Conteúdo salvo:")
!ls -la "{CAMINHO_SALVAR}"

import json
resumo = {
    "modelo": MODEL_CHECKPOINT,
    "config_split": CONFIG_VENCEDORA,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "f1_macro_teste": float(resultado_teste["eval_f1_macro"]),
    "acuracia_teste": float(resultado_teste["eval_accuracy"]),
    "melhor_f1_validacao": float(trainer_final.state.best_metric),
}
with open(CAMINHO_SALVAR + "resumo_treino.json", "w", encoding="utf-8") as f:
    json.dump(resumo, f, ensure_ascii=False, indent=2)

print("\nResumo salvo em resumo_treino.json:")
print(json.dumps(resumo, ensure_ascii=False, indent=2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo e tokenizer salvos em: /content/drive/MyDrive/Agente Mecânico/modelos/debertinha_mao_na_roda/
Conteúdo salvo:
total 163219
drwx------ 2 root root      4096 Sep 15 23:22 .
drwx------ 6 root root      4096 Sep 15 23:22 ..
-rw------- 1 root root      1396 Sep 15 23:22 config.json
-rw------- 1 root root 163812768 Sep 15 23:22 model.safetensors
-rw------- 1 root root       585 Sep 15 23:22 tokenizer_config.json
-rw------- 1 root root   3306338 Sep 15 23:22 tokenizer.json
-rw------- 1 root root      5165 Sep 15 23:22 training_args.bin

Resumo salvo em resumo_treino.json:
{
  "modelo": "sagui-nlp/debertinha-ptbr-xsmall",
  "config_split": "80_10_10",
  "seed": 42,
  "max_length": 64,
  "f1_macro_teste": 0.8783595429147415,
  "acuracia_teste": 0.8782894736842105,
  "melhor_f1_validacao": 0.8622318234700745
}
